### **Semana 5 - zero-shot, recuperación imagen-texto, métricas y análisis comparativo con OpenCLIP**

Este cuaderno está diseñado para
- evaluar clasificación **zero-shot**
- medir **image->text** y **text->image retrieval**
- comparar **prompt templates** y **prompt ensembles**
- comparar **checkpoints** abiertos de OpenCLIP
- construir una vía práctica con **FAISS**
- dejar un puente operativo hacia `torchrun` y SLURM


#### **Requisitos operativos**

Este proyecto está pensado para correrse dentro de tu contenedor Docker del curso y con una GPU local. La secuencia práctica es:

```bash
cd /workspace/Semana5
pip install -r requirements-extra.txt
python scripts/00_verify_env.py
bash scripts/run_local_pipeline.sh
```


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.dataset_utils import load_metadata, build_caption_targets
from src.openclip_utils import create_model, encode_texts
from src.metrics import compute_similarity_matrix, summarize_multi_target_ranking
from src.retrieval import topk_text_to_image, topk_image_to_text
from src.visualize import show_gallery


In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
metadata_csv = PROJECT_ROOT / 'data/bootstrap_flickr30k/metadata.csv'
prompt_json = PROJECT_ROOT / 'data/bootstrap_flickr30k/prompt_config.json'
emb_path = PROJECT_ROOT / 'outputs/embeddings/bootstrap_embeddings.npz'

metadata = load_metadata(metadata_csv, root=PROJECT_ROOT)
caption_df, image_to_text, text_to_image = build_caption_targets(metadata)
metadata.head()


#### **1. Dataset bootstrap real**

Se usa un subconjunto real de Flickr30k ya incorporado en el proyecto. Cada imagen tiene una caption principal, una etiqueta operativa para zero-shot y la lista completa de captions para evaluar retrieval multicaption.


In [ ]:
show_gallery(metadata, ncols=3, figsize=(12, 8));
plt.show()


#### **2. Construcción o carga de embeddings**

Si ya corriste `bash scripts/run_local_pipeline.sh`, el archivo `outputs/embeddings/bootstrap_embeddings.npz` ya existe. Si no existe, construimos embeddings aquí mismo.


In [ ]:
import subprocess

if not emb_path.exists():
    cmd = [
        sys.executable, 'scripts/02_build_embeddings.py',
        '--metadata-csv', str(metadata_csv),
        '--model-name', 'ViT-B-32',
        '--pretrained', 'laion2b_s34b_b79k',
        '--output', str(emb_path),
    ]
    subprocess.run(cmd, check=True)

bundle = np.load(emb_path, allow_pickle=True)
image_features = bundle['image_features']
text_features = bundle['text_features']
print('image_features:', image_features.shape)
print('text_features:', text_features.shape)


#### **3. Retrieval multicaption**

En Semana 5 conviene medir retrieval con la estructura correcta del problema:

- **image->text**: una imagen puede tener varias captions correctas.
- **text->image**: cada caption apunta a una imagen correcta.


In [ ]:
sim = compute_similarity_matrix(image_features, text_features)
text_positive = {idx: [img_idx] for idx, img_idx in text_to_image.items()}

image_to_text_metrics = summarize_multi_target_ranking(sim, image_to_text)
text_to_image_metrics = summarize_multi_target_ranking(sim.T, text_positive)

pd.DataFrame([
    {'direction': 'image_to_text', **{k: v for k, v in image_to_text_metrics.items() if k != 'Ranks'}},
    {'direction': 'text_to_image', **{k: v for k, v in text_to_image_metrics.items() if k != 'Ranks'}},
])


In [ ]:
# Ejemplo text -> image
model_name = str(bundle['model_name'])
pretrained = str(bundle['pretrained'])
model, _, tokenizer, device = create_model(model_name, pretrained)
query = 'people cooking in a kitchen'
q_feat = encode_texts(model, tokenizer, [query], device)
res = topk_text_to_image(q_feat, image_features, metadata, k=3)
res


In [ ]:
# Ejemplo image -> text
res_i2t = topk_image_to_text(4, sim, caption_df, k=5)
res_i2t


#### **4. Zero-shot con templates y prompt ensemble**

La evaluación zero-shot no debería limitarse a un solo prompt. Este cuaderno usa la configuración de `data/bootstrap_flickr30k/prompt_config.json` y compara plantillas individuales contra un **ensemble** de prompts.


In [ ]:
import subprocess
zs_summary = PROJECT_ROOT / 'outputs/metrics/zeroshot_prompt_summary.csv'
zs_preds = PROJECT_ROOT / 'outputs/metrics/zeroshot_predictions.csv'
zs_conf = PROJECT_ROOT / 'outputs/metrics/zeroshot_confusion.csv'

if not zs_summary.exists():
    cmd = [
        sys.executable, 'scripts/04_eval_zeroshot_prompt_ensembles.py',
        '--embeddings', str(emb_path),
        '--metadata-csv', str(metadata_csv),
        '--prompt-config', str(prompt_json),
        '--output-summary', str(zs_summary),
        '--output-predictions', str(zs_preds),
        '--output-confusion', str(zs_conf),
    ]
    subprocess.run(cmd, check=True)

pd.read_csv(zs_summary)


In [ ]:
pd.read_csv(zs_preds)


In [ ]:
conf = pd.read_csv(zs_conf, index_col=0)
conf


#### **5. Comparación de checkpoints**

Esta sección empuja el cuaderno a un nivel más propio:no solo usar un modelo, sino comparar varios checkpoints abiertos y discutir trade-offs entre retrieval y zero-shot.


In [ ]:
ckpt_cmp = PROJECT_ROOT / 'outputs/metrics/checkpoint_comparison.csv'
if not ckpt_cmp.exists():
    cmd = [
        sys.executable, 'scripts/05_compare_checkpoints.py',
        '--metadata-csv', str(metadata_csv),
        '--checkpoint-config', str(PROJECT_ROOT / 'configs/checkpoints.yaml'),
        '--prompt-config', str(prompt_json),
        '--output-csv', str(ckpt_cmp),
    ]
    subprocess.run(cmd, check=True)

pd.read_csv(ckpt_cmp)


#### **6. Búsqueda semántica con FAISS**

La parte de FAISS permite convertir el ejercicio de retrieval en un mini sistema de búsqueda realista.


In [ ]:
faiss_csv = PROJECT_ROOT / 'outputs/faiss/query_results.csv'
if not faiss_csv.exists():
    cmd = [
        sys.executable, 'scripts/06_build_faiss_index.py',
        '--embeddings', str(emb_path),
        '--metadata-csv', str(metadata_csv),
        '--queries-csv', str(PROJECT_ROOT / 'data/bootstrap_flickr30k/queries.csv'),
        '--output-csv', str(faiss_csv),
    ]
    subprocess.run(cmd, check=True)

pd.read_csv(faiss_csv).head(12)


#### **7. Ruta operativa realista**

Con una sola GPU local:
- construir embeddings
- medir retrieval y zero-shot
- comparar prompts y checkpoints
- usar FAISS
- hacer un fine-tuning corto sobre CSV solo como extensión

Con múltiples GPUs o cluster:
- migrar el fine-tuning a `torchrun`
- pasar a `slurm/finetune_openclip_csv_single_node.sbatch`
- o a `slurm/finetune_openclip_csv_multi_node.sbatch`

Estas plantillas siguen la lógica del README de OpenCLIP: `torchrun`, `dataset-type csv`, `dataset-type webdataset`, `--local-loss` y `--gather-with-grad`.


#### **8. Entregable de semana 5**

El entregable recomendable es un reporte breve con:

1. checkpoint y justificación
2. métricas de retrieval
3. accuracy zero-shot con al menos 3 templates
4. comparación template único vs ensemble
5. análisis de 5 errores
6. recomendación final sobre el mejor setup para hardware local
